In [1]:
import pandas as pd
from prepare_data import extract_gtcc, extract_mel_spectrogram, extract_mfcc, extract_lfcc, extract_cqcc
from gmm_bilstm_pipeline import evaluate_all_models, BiLSTM_model, create_dataloaders
from omegaconf import OmegaConf
from run_all_task import get_mixed_dataset

from ASV_dl_func import *

In [2]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [ ]:
config = OmegaConf.load("config.yaml")

la_metadata = config.datasets.LA.metadata
la_flac_folders = config.datasets.LA.flac
in_the_wild_dir = config.datasets.paths.in_the_wild_dir

In [ ]:
train_df, val_df, test_df = get_mixed_dataset(in_the_wild_dir, la_metadata, la_flac_folders)

In [22]:
train_aug = add_dataAugmentation(train_df)

In [ ]:
feature_extractors_map = {
    'cqcc': extract_cqcc,
    'gtcc': extract_gtcc,
    'mel-spect': extract_mel_spectrogram,
    'mfcc': extract_mfcc,
    'lfcc': extract_lfcc
}
col_name = feature_extractors_map.keys

In [8]:
train_df_prepared = extract_features(train_aug, feature_extractors_map)

   - Ekstrahuję: mel-spect


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:   15.8s
[Parallel(n_jobs=-1)]: Done 227 tasks      | elapsed:   16.7s
[Parallel(n_jobs=-1)]: Done 3680 tasks      | elapsed:   21.6s
[Parallel(n_jobs=-1)]: Done 9280 tasks      | elapsed:   27.3s
[Parallel(n_jobs=-1)]: Done 16480 tasks      | elapsed:   32.0s
[Parallel(n_jobs=-1)]: Done 25280 tasks      | elapsed:   37.4s
[Parallel(n_jobs=-1)]: Done 28600 out of 28631 | elapsed:   39.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 28631 out of 28631 | elapsed:   39.2s finished


In [9]:
train_df_prepared = train_df_prepared.dropna(subset=col_name)

In [11]:

val_df = extract_features(val_df, feature_extractors_map)
val_df_prepared = val_df.dropna()

test_df = extract_features(test_df, feature_extractors_map)
test_df_prepared = test_df.dropna()

   - Ekstrahuję: mel-spect


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 1120 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done 4535 tasks      | elapsed:    3.8s
[Parallel(n_jobs=-1)]: Done 4800 out of 4800 | elapsed:    4.0s finished


# Sprawdzenie zbalansowanai

In [12]:
train_df_noscale = balance_func(train_df_prepared, col_name='label')

Zbilansowane dane: true=8279, false=8279


In [13]:
val_df_noscale = balance_func(val_df_prepared, col_name='label')

Zbilansowane dane: true=1329, false=1329


# BiLSTM

In [16]:
cqcc_map = {"cqcc": extract_cqcc}

In [17]:
train_df_noscale = extract_features(train_aug, cqcc_map)
val_df_noscale = extract_features(val_df, cqcc_map)

   - Ekstrahuję: cqcc


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  52 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done 352 tasks      | elapsed:    7.2s
[Parallel(n_jobs=-1)]: Done 852 tasks      | elapsed:   17.4s
[Parallel(n_jobs=-1)]: Done 1552 tasks      | elapsed:   31.1s
[Parallel(n_jobs=-1)]: Done 2452 tasks      | elapsed:   48.2s
[Parallel(n_jobs=-1)]: Done 3552 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 4852 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done 5968 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done 6818 tasks      | elapsed:  2.5min
[Parallel(n_jobs=-1)]: Done 7872 tasks      | elapsed:  3.0min
[Parallel(n_jobs=-1)]: Done 9972 tasks      | elapsed:  3.4min
[Parallel(n_jobs=-1)]: Done 10111 out of 10126 | elapsed:  3.5min remaining:    0.2s
[Parallel(n_jobs=-1)]: Done 10126 out of 10126 | elapsed:  3.5min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers

   - Ekstrahuję: cqcc


[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:    4.7s
[Parallel(n_jobs=-1)]: Done 434 tasks      | elapsed:   11.6s
[Parallel(n_jobs=-1)]: Done 784 tasks      | elapsed:   20.6s
[Parallel(n_jobs=-1)]: Done 1234 tasks      | elapsed:   32.2s
[Parallel(n_jobs=-1)]: Done 1784 tasks      | elapsed:   46.4s
[Parallel(n_jobs=-1)]: Done 2000 out of 2000 | elapsed:   52.1s finished


In [28]:
cqcc_map = {"cqcc": extract_cqcc}

train_bilstm = filtr_nan(train_df_noscale)
train_bilstm['cqcc'] = train_bilstm['cqcc'].apply(transpose_cqcc)

test_bilstm = filtr_nan(val_df_noscale)
test_bilstm['cqcc'] = test_bilstm['cqcc'].apply(transpose_cqcc)

final_df = train_bilstm[train_bilstm['cqcc'].notnull()]
final_df_balanced = balance_func(final_df, col_name='label')

test_df = test_bilstm[test_bilstm['cqcc'].notnull()]
test_df_balanced = balance_func(test_df, col_name='label')

train_df, test_df, scaler = prepare_train_test_data(final_df_balanced, test_df=test_df_balanced, label_name='label')

all_results, val_loader = BiLSTM_model(train_df, test_df, num_epochs=100)

Zbilansowane dane: true=8263, false=8263
Zbilansowane dane: true=1329, false=1329

=== Trening: Adam, CrossEntropyLoss, LR=0.001, WD=1e-05, MOM=None ===
Folder: GMM-BiLSTM\Adam_CrossEntropyLoss_lr0_001_wd1e-05
Rozpoczęto trening BiLSTM (Adam, CrossEntropyLoss)...
Epoch 1/100 | Train Loss: 0.6640 | Val Loss: 0.6869 | Val Acc: 0.5767
Epoch 2/100 | Train Loss: 0.6890 | Val Loss: 0.6923 | Val Acc: 0.5237
Epoch 3/100 | Train Loss: 0.6518 | Val Loss: 0.5913 | Val Acc: 0.6930
Epoch 4/100 | Train Loss: 0.5924 | Val Loss: 0.4696 | Val Acc: 0.7844
Epoch 5/100 | Train Loss: 0.5429 | Val Loss: 0.4138 | Val Acc: 0.8236
Epoch 6/100 | Train Loss: 0.4792 | Val Loss: 0.3581 | Val Acc: 0.8480
Epoch 7/100 | Train Loss: 0.4191 | Val Loss: 0.4108 | Val Acc: 0.8251
Epoch 8/100 | Train Loss: 0.3737 | Val Loss: 0.3255 | Val Acc: 0.8702
Epoch 9/100 | Train Loss: 0.3276 | Val Loss: 0.3040 | Val Acc: 0.8789
Epoch 10/100 | Train Loss: 0.2954 | Val Loss: 0.3641 | Val Acc: 0.8668
Epoch 11/100 | Train Loss: 0.2510 |

In [ ]:
train_loader, val_loader = create_dataloaders(train_df, test_df)

In [32]:
evaluate_all_models("GMM-BiLSTM", train_loader=train_loader, train_df=train_df, test_df=test_df, test_loader=val_loader)

🔹 Wykryty input_dim = 19
✅ Wczytano model z: GMM-BiLSTM\AdamW_CrossEntropyLoss_lr0_0001_wd0_0001\biLstm_best_model.pt

--- Ewaluacja modelu: AdamW_CrossEntropyLoss_lr0_0001_wd0_0001 ---
Trening Gaussian Mixture (UBM)...
Initialization 0
  Iteration 10
  Iteration 20
  Iteration 30
  Iteration 40
  Iteration 50
  Iteration 60
  Iteration 70
  Iteration 80
  Iteration 90
  Iteration 100
Initialization did not converge.


C:\Users\Konrad\Desktop\IzaInz\AudioAnalysisDetector\.venv2\lib\site-packages\sklearn\mixture\_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(


Trening UBM zakończony w 1123.33 sekund.
Adaptacja GMM dla klas Genuine i DF...
Adaptacja MAP zakończona w 171.78 sekund.
Modele GMM zapisane w folderze 'GMM-BiLSTM/'.
Ewaluacja zakończona w 6.98 sekund.

--- Wyniki końcowe ---
Accuracy: 0.8942814145974417
F1: 0.8926251432938479
EER: 0.109104589917231
✅ Wczytano model z: GMM-BiLSTM\AdamW_CrossEntropyLoss_lr0_0001_wd1e-05\biLstm_best_model.pt

--- Ewaluacja modelu: AdamW_CrossEntropyLoss_lr0_0001_wd1e-05 ---
Trening Gaussian Mixture (UBM)...
Initialization 0
  Iteration 10
  Iteration 20
  Iteration 30
  Iteration 40
  Iteration 50
  Iteration 60
  Iteration 70
  Iteration 80
  Iteration 90
  Iteration 100
Initialization did not converge.


C:\Users\Konrad\Desktop\IzaInz\AudioAnalysisDetector\.venv2\lib\site-packages\sklearn\mixture\_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(


Trening UBM zakończony w 1160.30 sekund.
Adaptacja GMM dla klas Genuine i DF...
Adaptacja MAP zakończona w 173.78 sekund.
Modele GMM zapisane w folderze 'GMM-BiLSTM/'.
Ewaluacja zakończona w 6.96 sekund.

--- Wyniki końcowe ---
Accuracy: 0.8924003009781791
F1: 0.8915845337376801
EER: 0.10985703536493605
✅ Wczytano model z: GMM-BiLSTM\AdamW_CrossEntropyLoss_lr0_001_wd0_0001\biLstm_best_model.pt

--- Ewaluacja modelu: AdamW_CrossEntropyLoss_lr0_001_wd0_0001 ---
Trening Gaussian Mixture (UBM)...
Initialization 0
  Iteration 10
  Iteration 20
  Iteration 30
  Iteration 40
  Iteration 50
  Iteration 60
  Iteration 70
  Iteration 80
  Iteration 90
  Iteration 100
Initialization did not converge.


...

Trening UBM zakończony w 925.28 sekund.
Adaptacja GMM dla klas Genuine i DF...
Adaptacja MAP zakończona w 167.33 sekund.
Modele GMM zapisane w folderze 'GMM-BiLSTM/'.
Ewaluacja zakończona w 6.52 sekund.

--- Wyniki końcowe ---
Accuracy: 0.8980436418359669
F1: 0.8965253913707522
EER: 0.109104589917231
✅ Wczytano model z: GMM-BiLSTM\SGD_CrossEntropyLoss_lr0_001_wd1e-05_mom0_9\biLstm_best_model.pt

--- Ewaluacja modelu: SGD_CrossEntropyLoss_lr0_001_wd1e-05_mom0_9 ---
Trening Gaussian Mixture (UBM)...
Initialization 0
  Iteration 10
  Iteration 20
  Iteration 30
  Iteration 40
  Iteration 50
  Iteration 60
  Iteration 70
  Iteration 80
  Iteration 90
  Iteration 100
Initialization did not converge.


C:\Users\Konrad\Desktop\IzaInz\AudioAnalysisDetector\.venv2\lib\site-packages\sklearn\mixture\_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(


Trening UBM zakończony w 947.14 sekund.
Adaptacja GMM dla klas Genuine i DF...
Adaptacja MAP zakończona w 166.05 sekund.
Modele GMM zapisane w folderze 'GMM-BiLSTM/'.
Ewaluacja zakończona w 6.51 sekund.

--- Wyniki końcowe ---
Accuracy: 0.8950338600451467
F1: 0.89419795221843
EER: 0.10835214446952596
✅ Wczytano model z: GMM-BiLSTM\SGD_CrossEntropyLoss_lr0_001_wd1e-05_mom0_95\biLstm_best_model.pt

--- Ewaluacja modelu: SGD_CrossEntropyLoss_lr0_001_wd1e-05_mom0_95 ---
Trening Gaussian Mixture (UBM)...
Initialization 0
  Iteration 10
  Iteration 20
  Iteration 30
  Iteration 40
  Iteration 50
  Iteration 60
  Iteration 70
  Iteration 80
  Iteration 90
  Iteration 100
Initialization did not converge.


C:\Users\Konrad\Desktop\IzaInz\AudioAnalysisDetector\.venv2\lib\site-packages\sklearn\mixture\_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(


Trening UBM zakończony w 925.49 sekund.
Adaptacja GMM dla klas Genuine i DF...
Adaptacja MAP zakończona w 165.97 sekund.
Modele GMM zapisane w folderze 'GMM-BiLSTM/'.
Ewaluacja zakończona w 6.53 sekund.

--- Wyniki końcowe ---
Accuracy: 0.8954100827689992
F1: 0.8935681470137825
EER: 0.11136192626034612
